In [ ]:
%pip install -q "transformers>=4.55"  "huggingface_hub>=0.27"  "accelerate>=1.2"  "safetensors>=0.4"  "scikit-learn>=1.4"  "seaborn>=0.13" pandas matplotlib joblib

In [ ]:
import json
import platform
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import transformers
from huggingface_hub import login
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 100)

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sklearn": sklearn.__version__,
    "cuda": torch.cuda.is_available(),
})

In [ ]:
MODEL_ID = "google/gemma-3-270m-it"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"                                                  
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
model.to(DEVICE).eval()
model.requires_grad_(False)                                           

n_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Loaded: {MODEL_ID}")
print(f"Device: {DEVICE} | dtype: {DTYPE}")
print(f"Parameters: {n_params / 1e6:,.1f}M | currently marked trainable: {trainable_params / 1e6:,.1f}M")

In [ ]:
architecture = {
    "model_type": model.config.model_type,
    "layers": model.config.num_hidden_layers,
    "hidden_size": model.config.hidden_size,
    "attention_heads": model.config.num_attention_heads,
    "kv_heads": getattr(model.config, "num_key_value_heads", None),
    "vocab_size": model.config.vocab_size,
    "max_positions": getattr(model.config, "max_position_embeddings", None),
}
pd.Series(architecture, name="value").to_frame()

In [ ]:
messages = [{"role": "user", "content": "Explain a Transformer hidden state in two short sentences."}]
chat_inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(DEVICE)

with torch.inference_mode():
    generated = model.generate(
        **chat_inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

new_tokens = generated[0, chat_inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

In [ ]:
train_toxic = [
    "You are completely useless and nobody wants your help.",
    "Shut up, you pathetic fool.",
    "I will make you regret saying that.",
    "What an idiot; leave this group.",
    "People like you ruin everything.",
    "You are disgusting and should disappear.",
    "This is garbage, just like all your work.",
    "I hate you. Get lost.",
    "You are too stupid to understand anything.",
    "I hope you fail and everyone laughs at you.",
    "You deserve to be humiliated.",
    "Nobody cares about your worthless opinion.",
    "Go away; your presence makes everything worse.",
    "I will ruin your reputation for this.",
    "You are an embarrassment to the whole team.",
    "Stop talking, moron.",
]

train_nontoxic = [
    "I disagree, but I appreciate your perspective.",
    "This draft needs revision before we publish it.",
    "Could you explain how you reached that conclusion?",
    "I am frustrated by the delay, so let us reset the timeline.",
    "The calculation is incorrect; the second term needs a minus sign.",
    "Thank you for taking the time to review this.",
    "Please stop the deployment until the tests pass.",
    "The proposal does not meet the stated requirements.",
    "I would prefer a different approach to this problem.",
    "Your presentation was clear and useful.",
    "Let us discuss the disagreement calmly tomorrow.",
    "The service failed again, and we need to investigate why.",
    "I cannot approve this version because evidence is missing.",
    "We should invite another reviewer to reduce bias.",
    "That comment surprised me; can you clarify it?",
    "I value your help even though we reached different answers.",
]

test_toxic = [
    "Your ideas are worthless, so keep quiet.",
    "Everyone would be better off without you here.",
    "I am going to make your life miserable.",
    "Only a fool could produce work this bad.",
    "You are a disgrace and should leave.",
    "No one asked for your stupid opinion.",
    "I cannot stand you; just disappear.",
    "You will pay for embarrassing me.",
]

test_nontoxic = [
    "I strongly oppose this decision, but I respect the committee.",
    "Please remove this section because its evidence is weak.",
    "The result is disappointing; let us diagnose the failure.",
    "Could we pause and return to the original question?",
    "I do not think this method is reliable yet.",
    "Your answer contains an error in the final step.",
    "We need a safer plan before continuing.",
    "Thanks for the attempt; a second draft would help.",
]

def rows(texts, label, split):
    return [{"text": text, "label": label, "split": split} for text in texts]

df = pd.DataFrame(
    rows(train_toxic, 1, "train")
    + rows(train_nontoxic, 0, "train")
    + rows(test_toxic, 1, "test")
    + rows(test_nontoxic, 0, "test")
)
df["label_name"] = df["label"].map({0: "non-toxic", 1: "toxic"})
display(df.sample(8, random_state=SEED))
display(pd.crosstab(df["split"], df["label_name"], margins=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(data=df, x="split", hue="label_name", palette=["#2a9d8f", "#e76f51"], ax=axes[0])
axes[0].set_title("Balanced labels in each split")
axes[0].set_xlabel("")
axes[0].set_ylabel("examples")

lengths = df["text"].map(lambda text: len(tokenizer(text, add_special_tokens=True)["input_ids"]))
sns.histplot(x=lengths, hue=df["label_name"], multiple="dodge", bins=8, palette=["#2a9d8f", "#e76f51"], ax=axes[1])
axes[1].set_title("Token-length check")
axes[1].set_xlabel("tokens")
plt.tight_layout()
plt.show()

print("Length by label (a simple confound check):")
display(pd.DataFrame({"tokens": lengths, "label": df["label_name"]}).groupby("label")["tokens"].describe().round(2))

In [ ]:
def extract_last_token_representations(texts, batch_size=8, max_length=64):
    """Return [n_examples, n_hidden_state_points, hidden_size] float32 NumPy array."""
    batches = []
    for start in range(0, len(texts), batch_size):
        text_batch = texts[start:start + batch_size]
        encoded = tokenizer(
            text_batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.inference_mode():
            output = model(
                **encoded,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True,
            )

        last_position = encoded["attention_mask"].sum(dim=1) - 1
        batch_index = torch.arange(len(text_batch), device=DEVICE)
        pooled_by_layer = torch.stack(
            [state[batch_index, last_position] for state in output.hidden_states],
            dim=1,
        )
        batches.append(pooled_by_layer.float().cpu().numpy())

    return np.concatenate(batches, axis=0)

representations = extract_last_token_representations(df["text"].tolist())
expected_shape = (len(df), model.config.num_hidden_layers + 1, model.config.hidden_size)
assert representations.shape == expected_shape, (representations.shape, expected_shape)
assert np.isfinite(representations).all()

print("Representation tensor:", representations.shape)
print("Approximate cached size: %.1f MB" % (representations.nbytes / 2**20))

In [ ]:
norms = np.linalg.norm(representations, axis=-1)
norm_df = pd.DataFrame({
    "layer": np.tile(np.arange(norms.shape[1]), len(df)),
    "L2 norm": norms.ravel(),
    "label": np.repeat(df["label_name"].to_numpy(), norms.shape[1]),
})
plt.figure(figsize=(10, 4))
sns.lineplot(data=norm_df, x="layer", y="L2 norm", hue="label", estimator="mean", errorbar="sd")
plt.title("Residual-stream norm across depth (mean ± SD)")
plt.xlabel("hidden-state index: 0 = embeddings")
plt.tight_layout()
plt.show()

In [ ]:
train_mask = df["split"].eq("train").to_numpy()
test_mask = df["split"].eq("test").to_numpy()
y_train = df.loc[train_mask, "label"].to_numpy()
y_test = df.loc[test_mask, "label"].to_numpy()

def make_probe():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.03, penalty="l2", max_iter=4000, random_state=SEED),
    )

dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train), 1)), y_train)
dummy_accuracy = accuracy_score(y_test, dummy.predict(np.zeros((len(y_test), 1))))

rng = np.random.default_rng(SEED)
layer_rows = []
probes = {}

for layer in range(representations.shape[1]):
    X_train = representations[train_mask, layer, :]
    X_test = representations[test_mask, layer, :]

    probe = make_probe().fit(X_train, y_train)
    probes[layer] = probe
    probabilities = probe.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    shuffled_aurocs = []
    for _ in range(20):
        shuffled_probe = make_probe().fit(X_train, rng.permutation(y_train))
        shuffled_aurocs.append(roc_auc_score(y_test, shuffled_probe.predict_proba(X_test)[:, 1]))

    layer_rows.append({
        "layer": layer,
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "auroc": roc_auc_score(y_test, probabilities),
        "shuffle_auroc_mean": np.mean(shuffled_aurocs),
        "shuffle_auroc_sd": np.std(shuffled_aurocs),
        "dummy_accuracy": dummy_accuracy,
    })

layer_metrics = pd.DataFrame(layer_rows)
best_layer = int(layer_metrics.loc[layer_metrics["auroc"].idxmax(), "layer"])
display(layer_metrics.round(3))
print(f"Illustrative best layer by held-out AUROC: {best_layer}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].plot(layer_metrics["layer"], layer_metrics["auroc"], marker="o", label="true labels", color="#5b5bd6")
axes[0].plot(layer_metrics["layer"], layer_metrics["shuffle_auroc_mean"], marker=".", label="shuffled labels", color="#9ca3af")
axes[0].fill_between(
    layer_metrics["layer"],
    layer_metrics["shuffle_auroc_mean"] - layer_metrics["shuffle_auroc_sd"],
    layer_metrics["shuffle_auroc_mean"] + layer_metrics["shuffle_auroc_sd"],
    color="#9ca3af", alpha=0.18,
)
axes[0].axhline(0.5, linestyle="--", color="black", linewidth=1, label="chance AUROC")
axes[0].axvline(best_layer, linestyle=":", color="#e76f51", linewidth=2)
axes[0].set(title="Where is toxicity linearly decodable?", xlabel="hidden-state index", ylabel="held-out AUROC", ylim=(0, 1.03))
axes[0].legend(loc="lower right")

axes[1].plot(layer_metrics["layer"], layer_metrics["accuracy"], marker="o", label="accuracy", color="#2a9d8f")
axes[1].plot(layer_metrics["layer"], layer_metrics["f1"], marker="s", label="F1", color="#e76f51")
axes[1].axhline(dummy_accuracy, linestyle="--", color="#6b7280", label="dummy accuracy")
axes[1].set(title="Thresholded performance", xlabel="hidden-state index", ylabel="score", ylim=(0, 1.03))
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
best_probe = probes[best_layer]
X_test_best = representations[test_mask, best_layer, :]
test_probability = best_probe.predict_proba(X_test_best)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

prediction_table = df.loc[test_mask, ["text", "label", "label_name"]].copy()
prediction_table["p(toxic)"] = test_probability
prediction_table["predicted"] = np.where(test_prediction == 1, "toxic", "non-toxic")
prediction_table["correct"] = prediction_table["label"].to_numpy() == test_prediction
display(prediction_table.sort_values("p(toxic)", ascending=False).style.format({"p(toxic)": "{:.3f}"}))

print(classification_report(y_test, test_prediction, target_names=["non-toxic", "toxic"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, test_prediction, display_labels=["non-toxic", "toxic"], cmap="Blues")
plt.title(f"Held-out confusion matrix · layer {best_layer}")
plt.show()

In [ ]:
X_best_all = representations[:, best_layer, :]
X_scaled_all = StandardScaler().fit_transform(X_best_all)                                                   
projection = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled_all)
pca_df = pd.DataFrame({
    "PC1": projection[:, 0],
    "PC2": projection[:, 1],
    "label": df["label_name"],
    "split": df["split"],
})

plt.figure(figsize=(8, 5.5))
sns.scatterplot(
    data=pca_df,
    x="PC1", y="PC2", hue="label", style="split", s=105,
    palette={"non-toxic": "#2a9d8f", "toxic": "#e76f51"},
)
plt.title(f"PCA view of layer {best_layer} representations")
plt.tight_layout()
plt.show()

In [ ]:
def token_layer_probe_scores(text):
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
    with torch.inference_mode():
        output = model(
            **encoded,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    token_ids = encoded["input_ids"][0].detach().cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    score_rows = []
    for layer, state in enumerate(output.hidden_states):
        token_vectors = state[0].float().cpu().numpy()
        score_rows.append(probes[layer].predict_proba(token_vectors)[:, 1])
    return tokens, np.vstack(score_rows)

def plot_token_layer_map(text, max_display_tokens=28):
    tokens, scores = token_layer_probe_scores(text)
    tokens, scores = tokens[:max_display_tokens], scores[:, :max_display_tokens]
    clean_tokens = [token.replace("▁", "␠") for token in tokens]
    plt.figure(figsize=(max(10, 0.55 * len(clean_tokens)), 6))
    sns.heatmap(scores, vmin=0, vmax=1, cmap="magma", xticklabels=clean_tokens, yticklabels=np.arange(scores.shape[0]))
    plt.title("Probe-estimated toxicity of each accumulated token prefix")
    plt.xlabel("token (␠ marks a word boundary)")
    plt.ylabel("hidden-state index")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

plot_token_layer_map("I disagree with your calculation, but I appreciate the careful explanation.")
plot_token_layer_map("Your calculation is worthless, and only a fool would defend it.")

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(best_probe, ARTIFACT_DIR / "gemma3_270m_toxicity_linear_probe.joblib")
metadata = {
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "tokenizer_id": MODEL_ID,
    "hidden_state_index": best_layer,
    "hidden_state_semantics": "0=embedding output; k=output after transformer block k",
    "pooling": "last non-padding token",
    "max_length": 64,
    "positive_label": "interpersonal insult, demeaning dismissal, or threat",
    "decision_threshold": 0.5,
    "seed": SEED,
    "n_train": int(train_mask.sum()),
    "n_test": int(test_mask.sum()),
    "teaching_only": True,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "sklearn_version": sklearn.__version__,
}
(ARTIFACT_DIR / "probe_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(json.dumps(metadata, indent=2))

In [ ]:
def _find_module(root, dotted_paths):
    """Return the first available module path; robust to Hugging Face wrapper names."""
    for dotted_path in dotted_paths:
        module = root
        for part in dotted_path.split("."):
            if not hasattr(module, part):
                break
            module = getattr(module, part)
        else:
            return module
    raise AttributeError(f"None of these module paths exists: {dotted_paths}")

                                                                           
                                                                           
final_norm = _find_module(model, ["model.norm", "model.language_model.norm"])
unembed = _find_module(model, ["lm_head", "model.lm_head", "model.language_model.lm_head"])

@torch.inference_mode()
def logit_lens(text, top_k=8, max_length=64):
    """Return every layer's provisional next-token probabilities for one prompt."""
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    output = model(**encoded, output_hidden_states=True, use_cache=False, return_dict=True)
    last_position = int(encoded["attention_mask"].sum().item() - 1)
    probabilities, top_ids = [], []
    for residual in output.hidden_states:
        logits = unembed(final_norm(residual))[0, last_position].float()
        layer_probs = torch.softmax(logits, dim=-1)
        probabilities.append(layer_probs.cpu())
        top_ids.append(torch.topk(layer_probs, k=top_k).indices.cpu())
    return torch.stack(probabilities).numpy(), torch.stack(top_ids).numpy()

def _display_token(token_id):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    return token.replace("▁", "<space>").replace("Ġ", "<space>")

def plot_logit_lens(text, top_k=8, max_tokens=14):
    probabilities, top_ids = logit_lens(text, top_k=top_k)
    candidate_ids = np.unique(top_ids.ravel())
    peaks = probabilities[:, candidate_ids].max(axis=0)
    candidate_ids = candidate_ids[np.argsort(peaks)[-max_tokens:]][::-1]
    candidate_tokens = [_display_token(token_id) for token_id in candidate_ids]
    shown_probs = probabilities[:, candidate_ids]

    fig, axes = plt.subplots(1, 2, figsize=(15, max(4.5, 0.36 * len(candidate_ids) + 2)))
    sns.heatmap(
        np.log10(np.clip(shown_probs, 1e-9, None)).T, cmap="viridis",
        yticklabels=candidate_tokens, xticklabels=np.arange(probabilities.shape[0]),
        ax=axes[0], cbar_kws={"label": "log10 probability"},
    )
    axes[0].set(title="Logit lens: provisional next-token probabilities", xlabel="hidden-state index", ylabel="candidate next token")

    top_probability = probabilities.max(axis=1)
    entropy = -(probabilities * np.log2(np.clip(probabilities, 1e-12, None))).sum(axis=1)
    axes[1].plot(top_probability, marker="o", color="#5b5bd6", label="top-token probability")
    entropy_axis = axes[1].twinx()
    entropy_axis.plot(entropy, marker="s", color="#e76f51", label="distribution entropy")
    axes[1].set(title="Logit-lens confidence across depth", xlabel="hidden-state index", ylabel="top-token probability", ylim=(0, 1))
    entropy_axis.set_ylabel("entropy (bits)")
    handles, labels = axes[1].get_legend_handles_labels()
    handles2, labels2 = entropy_axis.get_legend_handles_labels()
    axes[1].legend(handles + handles2, labels + labels2, loc="best")
    fig.suptitle(f"Prompt: {text!r}", y=1.02, fontsize=11)
    plt.tight_layout()
    plt.show()

plot_logit_lens("Your calculation is worthless, and only a fool would defend it.")

In [ ]:
def probe_direction_in_residual_space(probe):
    """Undo StandardScaler so w acts directly on model residual vectors."""
    scaler = probe.named_steps["standardscaler"]
    classifier = probe.named_steps["logisticregression"]
    return classifier.coef_[0] / scaler.scale_

def weight_lens_scores(probe):
    direction = torch.tensor(probe_direction_in_residual_space(probe), device=DEVICE, dtype=torch.float32)
    vocabulary_vectors = unembed.weight.detach().float().to(DEVICE)
    return torch.nn.functional.cosine_similarity(vocabulary_vectors, direction.unsqueeze(0), dim=1).cpu().numpy()

def plot_weight_lens(layer, top_k=10, comparison_layers=None):
    if comparison_layers is None:
        comparison_layers = sorted(set([0, best_layer, representations.shape[1] - 1]))
    scores = weight_lens_scores(probes[layer])
    positive_ids = np.argsort(scores)[-top_k:][::-1]
    negative_ids = np.argsort(scores)[:top_k]
    selected_ids = np.concatenate([negative_ids[::-1], positive_ids])
    selected_labels = [_display_token(token_id) for token_id in selected_ids]

    fig, axes = plt.subplots(1, 2, figsize=(15, max(4.8, 0.35 * len(selected_ids) + 1.8)), gridspec_kw={"width_ratios": [1, 1.25]})
    colors = ["#2a9d8f"] * top_k + ["#e76f51"] * top_k
    axes[0].barh(selected_labels, scores[selected_ids], color=colors)
    axes[0].axvline(0, color="black", linewidth=0.8)
    axes[0].set(title=f"Weight lens at layer {layer}", xlabel="cosine alignment with probe direction", ylabel="vocabulary token")

    comparison = np.vstack([weight_lens_scores(probes[current_layer])[selected_ids] for current_layer in comparison_layers])
    sns.heatmap(comparison.T, center=0, cmap="vlag", yticklabels=selected_labels, xticklabels=comparison_layers, ax=axes[1], cbar_kws={"label": "cosine alignment"})
    axes[1].set(title="Same vocabulary directions at selected depths", xlabel="probe hidden-state index", ylabel="vocabulary token")
    fig.text(0.18, 0.01, "top: toxic-aligned     bottom: non-toxic-aligned", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()

plot_weight_lens(best_layer)

In [ ]:
import zipfile
from competition_probe import fit_competition_ensemble

                                                                                  
                                                              
X_competition = representations[train_mask, best_layer, :]
y_competition = y_train
groups_competition = None
competition_artifact = fit_competition_ensemble(
    X_competition, y_competition, groups=groups_competition, seed=SEED, repeats=10
)
print("Selected probes:", competition_artifact["model_names"])
print("Weights:", np.round(competition_artifact["weights"], 3))
print("OOF multi-metric score:", round(competition_artifact["oof_score"], 4))
print("OOF threshold:", round(competition_artifact["threshold"], 3))

SUBMISSION_DIR = Path("submission_build")
SUBMISSION_DIR.mkdir(exist_ok=True)

                                                                    
submission_probe_path = SUBMISSION_DIR / "trained_probe.joblib"
joblib.dump(competition_artifact, submission_probe_path)

                                                                                       
classifier_source = """import numpy as np
import joblib
from pathlib import Path


class Classifier:
    def __init__(self):
        model_path = Path(__file__).with_name("trained_probe.joblib")
        self.artifact = joblib.load(model_path)

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        expected = self.artifact["n_features"]
        if X.ndim != 2 or X.shape[1] != expected:
            raise ValueError(f"Expected [n, {expected}] embeddings; got {X.shape}")
        probability = np.zeros(len(X), dtype=float)
        for weight, model in zip(self.artifact["weights"], self.artifact["models"]):
            if hasattr(model, "predict_proba"):
                model_probability = model.predict_proba(X)[:, 1]
            else:
                score = np.clip(model.decision_function(X), -40, 40)
                model_probability = 1.0 / (1.0 + np.exp(-score))
            probability += weight * model_probability
        return (probability >= self.artifact["threshold"]).astype(int)
"""
(SUBMISSION_DIR / "classifier.py").write_text(classifier_source, encoding="utf-8")

                                                             
submission_zip = Path("submission.zip")
with zipfile.ZipFile(submission_zip, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(SUBMISSION_DIR / "classifier.py", arcname="classifier.py")
    archive.write(submission_probe_path, arcname="trained_probe.joblib")

with zipfile.ZipFile(submission_zip, mode="r") as archive:
    files_in_zip = archive.namelist()
    expected_files = ["classifier.py", "trained_probe.joblib"]
    assert files_in_zip == expected_files, f"Unexpected ZIP contents: {files_in_zip}"

print(f"Created: {submission_zip.resolve()}")
print(f"ZIP contents: {files_in_zip}")
print(f"Size: {submission_zip.stat().st_size / 1024:.1f} KiB")